##Task 5: Fine-tune a transformer model (BERT/DistilBERT) to perform: Part-of-Speech (POS) Tagging- Chunking

1. Install Packages

In [49]:
!pip install transformers datasets seqeval torch evaluate accelerate -q

2. Import Libraries

In [50]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
import evaluate
print("All imported!")

All imported!


 3. Load Dataset

In [51]:
dataset = load_dataset("conll2003")

print("Dataset loaded:")
print(dataset)

print("\nSample:")
print(dataset['train'][0])

Dataset loaded:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Sample:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


4. Check Labels

In [52]:
# POS labels from dataset
label_list = dataset["train"].features["pos_tags"].feature.names
num_labels = len(label_list)
print("POS Labels (", num_labels, "):", label_list[:10], "...")

# Mappings
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Sample mapping:", list(id2label.items())[:5])

POS Labels ( 47 ): ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``'] ...
Sample mapping: [(0, '"'), (1, "''"), (2, '#'), (3, '$'), (4, '(')]


5. Tokenization

In [54]:
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer ready.")

Tokenizer ready.


6. Data Preprocessing and Label Alignment

In [55]:
model_checkpoint = "distilbert-base-uncased" # Using DistilBERT for faster training
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special tokens like [SEP]
            new_labels.append(-100)
        else:
            # This is a subword! We assign -100 so the model ignores it in loss calculation
            new_labels.append(-100)
    return new_labels

def preprocess_function(examples):
    # We are focusing on 'pos_tags' for this specific fine-tuning run
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    all_labels = examples["pos_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

# Map the preprocessing over the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

7. Model Setup

In [57]:
# Creating label mappings
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


8. Training

In [60]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./bert-pos-tagger",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.246509,0.261191
2,0.190652,0.233240
3,0.162409,0.225250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.30532004005907154, metrics={'train_runtime': 332.0361, 'train_samples_per_second': 126.863, 'train_steps_per_second': 7.933, 'total_flos': 510472720030266.0, 'train_loss': 0.30532004005907154, 'epoch': 3.0})

9. Evaluation and Inference

In [65]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index -100
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Run Evaluation
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# --- Inference ---
from transformers import pipeline

pos_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text = "John works at Google in California"
results = pos_pipeline(text)

print("\n--- Inference Results ---")
for res in results:
    print(f"Word: {res['word']}, Tag: {res['entity_group']}")

Evaluation Results: {'eval_loss': 0.22525019943714142, 'eval_runtime': 6.1039, 'eval_samples_per_second': 532.447, 'eval_steps_per_second': 33.421, 'epoch': 3.0}

--- Inference Results ---
Word: john, Tag: NNP
Word: works, Tag: VBZ
Word: at, Tag: IN
Word: google, Tag: NNP
Word: in, Tag: IN
Word: california, Tag: NNP


##Comparison (POS Tagging vs. Chunking)

| Feature | POS Tagging | Chunking |
| :--- | :--- | :--- |
| **Focus** | Individual word grammar. | Grouping words into phrases. |
| **Difficulty** | Easier (local context). | Medium (boundary detection). |
| **Labeling** | One tag per word (e.g., NOUN). | BIO tagging (e.g., B-NP, I-NP). |

##Report & Observations

**1. Observations:** The model learns POS tagging very efficiently because word patterns are often consistent. Chunking is slightly more complex as it requires the model to understand where a noun phrase begins and ends.

**2. Challenges:** Handling subwords was the biggest challenge. If a word is split into 'play' and '##ing', only 'play' should have the label to avoid confusing the model.

**3. Insights:** Fine-tuning a pre-trained DistilBERT model is much faster and more accurate than building a model from scratch.